# 2.5 - Final Dataset Preparation

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Preparar el dataset final limpio para modelado:

1. **Cargar features completos** (step4 con clima)
2. **Limpieza consolidada de NaNs** (evitar repetir en cada modelo)
3. **Validación de calidad** (verificar consistencia temporal)
4. **Guardar dataset listo para modelado** con metadata completa

**Salida:**
- `features_final_modeling.csv` - Dataset limpio sin NaNs
- `metadata_final_dataset.json` - Documentación completa de limpieza

## Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
from datetime import datetime

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"✓ Período de análisis: {START_DATE} → {END_DATE}")

✓ Base directory: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Processed directory: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed
✓ Período de análisis: 2000-01-01 → 2025-11-10


## 1. Cargar Dataset Completo (Step 4 - Con Climate Features)

In [2]:
# Cargar dataset con todas las features generadas
df_features = pd.read_csv(PROCESSED_DIR / 'features_step4_climate.csv', index_col=0, parse_dates=True)

print("=" * 80)
print("DATASET FEATURES COMPLETO (Step 4)")
print("=" * 80)
print(f"Shape: {df_features.shape}")
print(f"Período: {df_features.index.min()} → {df_features.index.max()}")
print(f"Días: {len(df_features):,}")
print(f"Features: {len(df_features.columns):,}")
print(f"\nPrimeras columnas: {df_features.columns[:10].tolist()}")
print(f"Últimas columnas: {df_features.columns[-10:].tolist()}")

DATASET FEATURES COMPLETO (Step 4)
Shape: (6731, 3186)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00
Días: 6,731
Features: 3,186

Primeras columnas: ['Baltic_Dry_Index', 'Brent_Crude', 'Cocoa', 'Coffee', 'Copper', 'Corn', 'Cotton', 'Crude_Oil', 'Ethanol', 'Feeder_Cattle']
Últimas columnas: ['ET0_Global_Grain_change_rate7', 'GDD_Global_Grain_cumsum7', 'GDD_Global_Grain_cumsum30', 'GDD_Global_Grain_cumsum90', 'Heat_Stress_Days_cumsum7', 'Heat_Stress_Days_cumsum30', 'Heat_Stress_Days_cumsum90', 'Precip_Deficit_cumsum7', 'Precip_Deficit_cumsum30', 'Precip_Deficit_cumsum90']


## 2. Diagnóstico de Missing Values

Antes de limpiar, documentar estado inicial de NaNs por tipo de feature.

In [3]:
# Calcular missing values por columna
missing_summary = pd.DataFrame({
    'missing_count': df_features.isnull().sum(),
    'missing_pct': (df_features.isnull().sum() / len(df_features) * 100).round(2)
}).sort_values('missing_count', ascending=False)

missing_summary = missing_summary[missing_summary['missing_count'] > 0]

print("=" * 80)
print("MISSING VALUES POR FEATURE")
print("=" * 80)
print(f"\nTotal features con missing: {len(missing_summary)} / {len(df_features.columns)}")
print(f"Total missing values: {df_features.isnull().sum().sum():,}")
print(f"Porcentaje total: {(df_features.isnull().sum().sum() / df_features.size * 100):.2f}%\n")

if len(missing_summary) > 0:
    print("Top 20 features con más missing:")
    print(missing_summary.head(20))

MISSING VALUES POR FEATURE

Total features con missing: 2790 / 3186
Total missing values: 1,672,101
Porcentaje total: 7.80%

Top 20 features con más missing:
                                         missing_count  missing_pct
Baltic_Dry_Index_volume_simple_return30           6731        100.0
Heat_Stress_Days_simple_return1                   6731        100.0
Baltic_Dry_Index_volume_price_to_ma90             6731        100.0
Baltic_Dry_Index_volume_vol_ratio_7_30            6731        100.0
Baltic_Dry_Index_volume_vol_ratio_30_90           6731        100.0
Heat_Stress_Days_price_to_ma90                    6731        100.0
Heat_Stress_Days_simple_return90                  6731        100.0
Heat_Stress_Days_log_return90                     6731        100.0
Heat_Stress_Days_simple_return30                  6731        100.0
Heat_Stress_Days_log_return30                     6731        100.0
Heat_Stress_Days_simple_return7                   6731        100.0
Heat_Stress_Days_log_retur

In [4]:
# Clasificar missing por tipo de feature
def clasificar_feature(col_name):
    """Clasificar feature por su nombre para análisis de missing."""
    if '_lag_' in col_name:
        return 'Temporal Lags'
    elif '_roll_' in col_name or '_ma_' in col_name or '_ema_' in col_name:
        return 'Rolling Stats'
    elif '_return_' in col_name or '_vol_' in col_name or '_bb_' in col_name:
        return 'Returns & Volatility'
    elif col_name.startswith('temp_') or col_name.startswith('prec_') or col_name in ['oni', 'et0_global', 'gdd_30d_global', 'heat_stress_days_30d', 'prec_deficit_30d']:
        return 'Climate'
    else:
        return 'Base Features'

# Agrupar missing por tipo
missing_summary['feature_type'] = missing_summary.index.map(clasificar_feature)
missing_by_type = missing_summary.groupby('feature_type').agg({
    'missing_count': ['sum', 'mean', 'count'],
    'missing_pct': 'mean'
}).round(2)

print("\n" + "=" * 80)
print("MISSING VALUES POR TIPO DE FEATURE")
print("=" * 80)
print(missing_by_type)


MISSING VALUES POR TIPO DE FEATURE
                     missing_count               missing_pct
                               sum    mean count        mean
feature_type                                                
Base Features              1222686  609.51  2006        9.06
Returns & Volatility        449415  573.23   784        8.52


## 3. Estrategia de Limpieza de NaNs

**Principios:**
1. **Temporal Lags & Rolling Stats:** Forward fill (ffill) - usar valor anterior
2. **Returns & Volatility:** Median imputation (volatilidad histórica)
3. **Climate Features:** Median imputation (promedio climático)
4. **Base Features:** Ya no deberían tener missing (vienen de data/processed)

**Orden de aplicación:**
1. ffill para features temporales (lags, rolling)
2. Median imputation para features calculados (returns, volatility, clima)
3. Verificación final (assert no quedan NaNs)

In [5]:
# Crear copia para limpieza
df_clean = df_features.copy()

# Registrar operaciones de limpieza
cleaning_log = {
    'timestamp': datetime.now().isoformat(),
    'input_shape': df_features.shape,
    'total_missing_before': int(df_features.isnull().sum().sum()),
    'operations': []
}

print("=" * 80)
print("INICIANDO LIMPIEZA DE NaNs")
print("=" * 80)
print(f"Missing inicial: {df_features.isnull().sum().sum():,} ({(df_features.isnull().sum().sum() / df_features.size * 100):.2f}%)")

INICIANDO LIMPIEZA DE NaNs
Missing inicial: 1,672,101 (7.80%)


### 3.1 Forward Fill para Features Temporales

In [6]:
# Identificar columnas temporales (lags y rolling)
temporal_cols = [col for col in df_clean.columns if '_lag_' in col or '_roll_' in col or '_ma_' in col or '_ema_' in col]

print(f"\n[1] Forward Fill para {len(temporal_cols)} features temporales")
missing_before_ffill = df_clean[temporal_cols].isnull().sum().sum()

# Aplicar ffill
df_clean[temporal_cols] = df_clean[temporal_cols].fillna(method='ffill')

missing_after_ffill = df_clean[temporal_cols].isnull().sum().sum()
print(f"    Missing eliminado: {missing_before_ffill:,} → {missing_after_ffill:,} (Δ = {missing_before_ffill - missing_after_ffill:,})")

# Registrar operación
cleaning_log['operations'].append({
    'step': 1,
    'method': 'forward_fill',
    'columns': temporal_cols,
    'missing_before': int(missing_before_ffill),
    'missing_after': int(missing_after_ffill)
})


[1] Forward Fill para 0 features temporales
    Missing eliminado: 0.0 → 0.0 (Δ = 0.0)


C:\Users\trico\AppData\Local\Temp\ipykernel_18920\2158186144.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean[temporal_cols] = df_clean[temporal_cols].fillna(method='ffill')


### 3.2 Median Imputation para Features Calculados

In [7]:
# Identificar columnas para median imputation (todo excepto temporales)
median_cols = [col for col in df_clean.columns if col not in temporal_cols]

print(f"\n[2] Median Imputation para {len(median_cols)} features calculados")
missing_before_median = df_clean[median_cols].isnull().sum().sum()

# Aplicar median imputation
for col in median_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        
        # Si la columna es toda NaN (edge case), usar 0
        if pd.isna(median_val):
            median_val = 0
            print(f"    WARNING: {col} es toda NaN, usando 0")
        
        df_clean[col].fillna(median_val, inplace=True)

missing_after_median = df_clean[median_cols].isnull().sum().sum()
print(f"    Missing eliminado: {missing_before_median:,} → {missing_after_median:,} (Δ = {missing_before_median - missing_after_median:,})")

# Registrar operación
cleaning_log['operations'].append({
    'step': 2,
    'method': 'median_imputation',
    'columns': median_cols,
    'missing_before': int(missing_before_median),
    'missing_after': int(missing_after_median)
})


[2] Median Imputation para 3186 features calculados


C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_val, inplace=True)
C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_val, inplace=True)
C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_val, inplace=True)
C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_val, inplace=True)
C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For examp

    Missing eliminado: 1,672,101 → 0 (Δ = 1,672,101)


f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\trico\AppData\Local\Temp\ipykernel_18920\838850452.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean[col].fillna(median_val, inplace=True)
f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\nanfunctions.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\trico\AppData\Local\Temp\ipykernel_

### 3.3 Verificación Final

In [8]:
# Verificar que no quedan NaNs
total_missing_final = df_clean.isnull().sum().sum()
print("\n" + "=" * 80)
print("VERIFICACIÓN FINAL")
print("=" * 80)
print(f"Missing después de limpieza: {total_missing_final:,}")
print(f"Shape: {df_clean.shape}")
print(f"Período: {df_clean.index.min()} → {df_clean.index.max()}")

# Assert crítico
assert total_missing_final == 0, f"ERROR: Todavía quedan {total_missing_final:,} NaNs después de limpieza"
print("\n✓ DATASET LIMPIO - Sin missing values")

# Actualizar log
cleaning_log['total_missing_after'] = int(total_missing_final)
cleaning_log['output_shape'] = df_clean.shape


VERIFICACIÓN FINAL
Missing después de limpieza: 0
Shape: (6731, 3186)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00

✓ DATASET LIMPIO - Sin missing values


## 4. Validación de Calidad del Dataset Final

### 4.1 Verificar Continuidad Temporal

In [9]:
# Verificar que no hay gaps en el índice temporal
date_range = pd.date_range(start=df_clean.index.min(), end=df_clean.index.max(), freq='D')
missing_dates = date_range.difference(df_clean.index)

print("=" * 80)
print("VALIDACIÓN DE CONTINUIDAD TEMPORAL")
print("=" * 80)
print(f"Fecha inicial: {df_clean.index.min()}")
print(f"Fecha final: {df_clean.index.max()}")
print(f"Días esperados: {len(date_range):,}")
print(f"Días en dataset: {len(df_clean):,}")
print(f"Fechas faltantes: {len(missing_dates)}")

if len(missing_dates) > 0:
    print(f"\nWARNING: {len(missing_dates)} fechas faltantes:")
    print(missing_dates[:10])  # Mostrar primeras 10
else:
    print("\n✓ Serie temporal continua sin gaps")

VALIDACIÓN DE CONTINUIDAD TEMPORAL
Fecha inicial: 2000-01-03 00:00:00
Fecha final: 2025-11-10 00:00:00
Días esperados: 9,444
Días en dataset: 6,731
Fechas faltantes: 2713

DatetimeIndex(['2000-01-08', '2000-01-09', '2000-01-15', '2000-01-16', '2000-01-22', '2000-01-23', '2000-01-29', '2000-01-30', '2000-02-05', '2000-02-06'], dtype='datetime64[ns]', freq=None)


### 4.2 Verificar Ranges de Variables

In [10]:
# Verificar que no hay valores infinitos o extremos anómalos
print("\n" + "=" * 80)
print("VALIDACIÓN DE RANGES")
print("=" * 80)

# Infinitos
inf_counts = np.isinf(df_clean).sum().sum()
print(f"Valores infinitos: {inf_counts}")

if inf_counts > 0:
    inf_cols = df_clean.columns[np.isinf(df_clean).sum() > 0]
    print(f"Columnas con infinitos: {inf_cols.tolist()}")

# Estadísticas básicas
print("\nEstadísticas básicas del dataset:")
print(df_clean.describe().iloc[:3])  # Solo count, mean, std


VALIDACIÓN DE RANGES
Valores infinitos: 53788
Columnas con infinitos: ['ONI_price_to_ma30', 'Crude_Oil_log_return1', 'Crude_Oil_simple_return1', 'Crude_Oil_log_return7', 'Crude_Oil_simple_return7', 'Crude_Oil_log_return30', 'Crude_Oil_simple_return30', 'Crude_Oil_log_return90', 'Crude_Oil_simple_return90', 'Brent_Crude_volume_log_return1', 'Brent_Crude_volume_simple_return1', 'Brent_Crude_volume_log_return7', 'Brent_Crude_volume_simple_return7', 'Brent_Crude_volume_log_return30', 'Brent_Crude_volume_simple_return30', 'Brent_Crude_volume_log_return90', 'Brent_Crude_volume_simple_return90', 'Cocoa_volume_log_return1', 'Cocoa_volume_simple_return1', 'Cocoa_volume_log_return7', 'Cocoa_volume_simple_return7', 'Cocoa_volume_log_return30', 'Cocoa_volume_simple_return30', 'Cocoa_volume_log_return90', 'Cocoa_volume_simple_return90', 'Coffee_volume_log_return1', 'Coffee_volume_simple_return1', 'Coffee_volume_log_return7', 'Coffee_volume_simple_return7', 'Coffee_volume_log_return30', 'Coffee_vol

f:\miniconda\envs\ds\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
f:\miniconda\envs\ds\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
f:\miniconda\envs\ds\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
f:\miniconda\envs\ds\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
f:\miniconda\envs\ds\Lib\site-packages\pandas\core\nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
f:\miniconda\envs\ds\Lib\site-packages\numpy\core\_methods.py:49: RuntimeWarning: invalid value encountered in reduce
  return umr_sum(a, 

       Baltic_Dry_Index  Brent_Crude        Cocoa       Coffee       Copper         Corn       Cotton    Crude_Oil      Ethanol  Feeder_Cattle         Gold  Heating_Oil    Lean_Hogs  Live_Cattle       Lumber  Natural_Gas          Oat    Palladium     Platinum  RBOB_Gasoline       Silver  Soybean_Meal  Soybean_Oil     Soybeans        Sugar  ...  psd_argentina_Production_vol_ratio_7_30  psd_argentina_Production_vol_ratio_30_90  psd_argentina_Exports_vol_ratio_7_30  psd_argentina_Exports_vol_ratio_30_90  psd_argentina_Stock_to_Use_Ratio_vol_ratio_7_30  psd_argentina_Stock_to_Use_Ratio_vol_ratio_30_90  psd_china_Imports_vol_ratio_7_30  psd_china_Imports_vol_ratio_30_90  psd_china_Crush_vol_ratio_7_30  psd_china_Crush_vol_ratio_30_90  psd_china_Ending_Stocks_vol_ratio_7_30  psd_china_Ending_Stocks_vol_ratio_30_90  psd_china_Stock_to_Use_Ratio_vol_ratio_7_30  psd_china_Stock_to_Use_Ratio_vol_ratio_30_90  Temp_Global_Grain_zscore  ET0_Global_Grain_change_rate7  GDD_Global_Grain_cumsum7  \
cou

### 4.3 Verificar Correlación con Targets

In [11]:
# Identificar columnas target (precios base de commodities)
target_candidates = [col for col in df_clean.columns if col in ['soy', 'corn', 'wheat', 'coffee', 'sugar']]

if len(target_candidates) > 0:
    print("\n" + "=" * 80)
    print("CORRELACIÓN CON TARGETS (Precios Base)")
    print("=" * 80)
    
    for target in target_candidates:
        # Calcular correlación con todas las features
        corr_with_target = df_clean.corr()[target].sort_values(ascending=False)
        
        print(f"\n{target.upper()} - Top 10 features más correlacionadas:")
        print(corr_with_target.head(10))
else:
    print("\nWARNING: No se encontraron targets en el dataset")

## 5. Guardar Dataset Final para Modelado

In [12]:
# Guardar dataset limpio
output_path = PROCESSED_DIR / 'features_final_modeling.csv'
df_clean.to_csv(output_path)

print("=" * 80)
print("GUARDANDO DATASET FINAL")
print("=" * 80)
print(f"✓ Dataset guardado: {output_path}")
print(f"  Shape: {df_clean.shape}")
print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Missing values: 0 (verificado)")

GUARDANDO DATASET FINAL
✓ Dataset guardado: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\features_final_modeling.csv
  Shape: (6731, 3186)
  Size: 279.63 MB
  Missing values: 0 (verificado)


## 6. Guardar Metadata Completa

In [13]:
# Construir metadata completa del dataset final
metadata = {
    'dataset_name': 'features_final_modeling',
    'creation_date': datetime.now().isoformat(),
    'source': 'features_step4_climate.csv',
    'cleaning_applied': True,
    
    # Dimensiones
    'shape': {
        'rows': df_clean.shape[0],
        'columns': df_clean.shape[1]
    },
    
    # Período temporal
    'temporal_range': {
        'start': df_clean.index.min().isoformat(),
        'end': df_clean.index.max().isoformat(),
        'days': len(df_clean),
        'missing_dates': len(missing_dates)
    },
    
    # Calidad de datos
    'data_quality': {
        'missing_values': 0,
        'infinite_values': int(inf_counts),
        'total_cells': int(df_clean.size)
    },
    
    # Composición de features
    'feature_composition': {
        'total_features': len(df_clean.columns),
        'temporal_lags': len([c for c in df_clean.columns if '_lag_' in c]),
        'rolling_stats': len([c for c in df_clean.columns if '_roll_' in c or '_ma_' in c or '_ema_' in c]),
        'returns_volatility': len([c for c in df_clean.columns if '_return_' in c or '_vol_' in c or '_bb_' in c]),
        'climate': len([c for c in df_clean.columns if c.startswith('temp_') or c.startswith('prec_') or c in ['oni', 'et0_global', 'gdd_30d_global', 'heat_stress_days_30d', 'prec_deficit_30d']]),
        'base': len([c for c in df_clean.columns if c in ['soy', 'corn', 'wheat', 'coffee', 'sugar', 'crude_oil', 'gold', 'usd_index', 'vix', 'sp500']])
    },
    
    # Log de limpieza
    'cleaning_log': cleaning_log,
    
    # Target variables disponibles
    'targets_available': target_candidates,
    
    # Columnas
    'columns': df_clean.columns.tolist()
}

# Guardar metadata
metadata_path = PROCESSED_DIR / 'metadata_final_dataset.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✓ Metadata guardada: {metadata_path}")


✓ Metadata guardada: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\metadata_final_dataset.json


## Resumen Final

In [14]:
print("\n" + "=" * 80)
print("RESUMEN - DATASET FINAL PARA MODELADO")
print("=" * 80)

print("\n✓ DATASET LIMPIO Y VALIDADO")
print(f"  - Shape: {df_clean.shape[0]:,} días × {df_clean.shape[1]} features")
print(f"  - Período: {df_clean.index.min().date()} → {df_clean.index.max().date()}")
print(f"  - Missing values: 0")
print(f"  - Infinite values: {inf_counts}")

print("\n✓ COMPOSICIÓN DE FEATURES:")
print(f"  - Temporal Lags: {metadata['feature_composition']['temporal_lags']}")
print(f"  - Rolling Stats: {metadata['feature_composition']['rolling_stats']}")
print(f"  - Returns & Volatility: {metadata['feature_composition']['returns_volatility']}")
print(f"  - Climate: {metadata['feature_composition']['climate']}")
print(f"  - Base Features: {metadata['feature_composition']['base']}")

print("\n✓ LIMPIEZA APLICADA:")
print(f"  - Forward fill: {cleaning_log['operations'][0]['missing_before'] - cleaning_log['operations'][0]['missing_after']:,} NaNs eliminados")
print(f"  - Median imputation: {cleaning_log['operations'][1]['missing_before'] - cleaning_log['operations'][1]['missing_after']:,} NaNs eliminados")

print("\n✓ ARCHIVOS GENERADOS:")
print(f"  - {output_path.name}")
print(f"  - {metadata_path.name}")

print("\n" + "=" * 80)
print("DATASET LISTO PARA NOTEBOOKS DE MODELADO (3.x)")
print("=" * 80)
print("\nPróximos pasos:")
print("  1. Usar features_final_modeling.csv en notebooks 3.1, 3.2, 3.3, 3.4, 3.5")
print("  2. ELIMINAR limpieza de NaNs redundante en cada notebook de modelado")
print("  3. Cargar directamente sin necesidad de fillna/dropna adicional")


RESUMEN - DATASET FINAL PARA MODELADO

✓ DATASET LIMPIO Y VALIDADO
  - Shape: 6,731 días × 3186 features
  - Período: 2000-01-03 → 2025-11-10
  - Missing values: 0
  - Infinite values: 53788

✓ COMPOSICIÓN DE FEATURES:
  - Temporal Lags: 0
  - Rolling Stats: 0
  - Returns & Volatility: 784
  - Climate: 0
  - Base Features: 0

✓ LIMPIEZA APLICADA:
  - Forward fill: 0 NaNs eliminados
  - Median imputation: 1,672,101 NaNs eliminados

✓ ARCHIVOS GENERADOS:
  - features_final_modeling.csv
  - metadata_final_dataset.json

DATASET LISTO PARA NOTEBOOKS DE MODELADO (3.x)

Próximos pasos:
  1. Usar features_final_modeling.csv en notebooks 3.1, 3.2, 3.3, 3.4, 3.5
  2. ELIMINAR limpieza de NaNs redundante en cada notebook de modelado
  3. Cargar directamente sin necesidad de fillna/dropna adicional
